# Blue team: inspect, activate shadow inference, then validate real labels
Activation makes the scheduled worker load a local release; it does **not** enable reporting.
Restart this kernel before activation to release any training model. The service needs a matching immutable manifest and verified model checksums.
Actual reports require an account session token, own-account IDs, explicit reporting opt-in, an approved review candidate, and available durable team budgets.
Automatic approval additionally requires an independent real-label evaluation. Never infer outcomes from a missing/hidden post: temporary hiding is not a moderator ruling.


In [ ]:
from pathlib import Path

from botfluencer.blue.data import private_directory

# Copy this notebook into ~/botfluencer-notebooks before running it.
private_directory(Path.cwd())
DATA = Path("/mnt/d/botfluencer/data/blue-v1")
RELEASES = Path("/mnt/d/botfluencer/models/releases")
ACTIVE = Path("/mnt/d/botfluencer/models/blue-active")

In [ ]:
import json

from botfluencer.blue.model import Manifest

release = Path(json.loads((DATA / "last-release.json").read_text())["path"])
manifest = Manifest.model_validate_json((release / "manifest.json").read_text())
manifest.verify(release)
print(json.loads((release / "evaluation.json").read_text()))
print("Model version:", manifest.version)

In [ ]:
from botfluencer.blue.training import activate

ACTIVATE_SHADOW = False  # Set True after inspecting the evaluation.
if ACTIVATE_SHADOW:
    activate(release, ACTIVE)
    print("Shadow release activated. Resume the blue worker; reporting settings are unchanged.")

## Review candidates privately
This displays collected text in this private notebook only. Do not copy the executed notebook back into the repository.
Review the author and several posts, exclude your own accounts, and check the known-NPC archive. A confidence score is statistical evidence, not proof.
To approve one existing candidate later: `botfluencer blue approve --post-id ID` with the private environment file.


In [ ]:
from sqlalchemy import select

from botfluencer.config import Settings
from botfluencer.storage.database import Database
from botfluencer.storage.models import BlueReport, NPCPost


async def review():
    db = Database(Settings(_env_file=Path.home() / ".config/botfluencer/runtime.env"))
    try:
        async with db.sessions() as session:
            query = (
                select(
                    BlueReport.post_id,
                    BlueReport.author_id,
                    BlueReport.probability,
                    BlueReport.status,
                    NPCPost.post,
                )
                .join(
                    NPCPost,
                    (NPCPost.namespace == BlueReport.namespace)
                    & (NPCPost.post_id == BlueReport.post_id),
                )
                .where(BlueReport.namespace == "ctn-2026")
                .order_by(BlueReport.probability.desc())
                .limit(20)
            )
            return (await session.execute(query)).mappings().all()
    finally:
        await db.close()


# Uncomment to display text in this private notebook:
# await review()

## Independent real-player validation (later)
Create private JSONL rows with `text`, `label` (NPC=0, confirmed competitor=1), `group`=`author:ACCOUNT_ID`, and `source`=`team_review` or `moderator`.
Use actual independently reviewed examples, not synthetic Qwen outputs or the detector's own predictions. Use authors/text absent from training and calibration.
The helper rejects overlap and reports confidence intervals. Automatic reporting stays blocked unless there are at least 100 real positives, 400 negatives, a 95% lower precision bound ≥0.95 and 95% upper false-positive bound ≤0.01.
Record moderator rulings with `botfluencer blue outcome --post-id ID --ruling genuine|inauthentic`. A genuine ruling persists a conservative six-hour team block. Reports remain pending until a real outcome is recorded; default pending-report budget is one.


In [ ]:
from botfluencer.blue.training import validate_real

RUN_REAL_VALIDATION = False
if RUN_REAL_VALIDATION:
    print(validate_real(release, DATA / "real-reviewed-holdout.jsonl"))